# Retrieval-aided Candidate Pruning for GLiNER2

This notebook evaluates train-set-assisted candidate pruning for Banking77 intent classification. It is not pure zero-shot: the Banking77 train split is used only as a retrieval memory. No GLiNER2 parameters are updated, no LoRA is used, no paid API is called, and no manual annotations are created.

In [ ]:
PROJECT_NAME = "gliner2_retrieval_candidate_pruning"
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "mteb/banking77"
MODE = "small"  # smoke / small / full
USE_GPU_IF_AVAILABLE = True
SEED = 42
SMOKE_N_EXAMPLES = 20
SMALL_PER_LABEL = 5
FULL_USE_ALL_TEST = True
CONFIRM_FULL_RUN = False
FORCE_RERUN = False
OUTPUT_DIR = "/content/gliner2_schema_outputs"
RETRIEVAL_OUTPUT_SUBDIR = "retrieval_pruning"

TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 1
TFIDF_MAX_FEATURES = 50000
K_NEIGHBORS = 15
EXPANDED_K_VALUES = [15, 25, 50]
MIN_CANDIDATES = 5
MAX_CANDIDATES = 15

if MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "MODE='full' requires CONFIRM_FULL_RUN=True. This prevents accidental long runs."
    )

## Install and import

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR) / RETRIEVAL_OUTPUT_SUBDIR
PREDICTIONS_DIR = OUTPUT_PATH / "predictions"
TABLES_DIR = OUTPUT_PATH / "tables"
FIGURES_DIR = OUTPUT_PATH / "figures"
for path in [OUTPUT_PATH, PREDICTIONS_DIR, TABLES_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()
    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    repo_url = "https://github.com/daidai-su/GLiNER2-demo.git"
    clone_dir = Path("/content/GLiNER2-demo")
    print("Project files were not found in the runtime.")
    print(f"Trying to clone project files from {repo_url} ...")
    try:
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
        PROJECT_ROOT = find_project_root()
    except Exception as exc:
        print(f"Automatic clone failed: {exc}")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Upload or clone the full repository into Colab."
    )

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
)
print("Dependencies installed.")

## Environment check

In [ ]:
import hashlib
import random
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from gliner2_project.analysis import (
    compare_method_examples,
    method_metrics_frame,
    per_label_delta_frame,
    per_label_metrics_frame,
)
from gliner2_project.candidate_pruning import (
    build_retrieved_candidate_set,
    candidate_recall_row,
)
from gliner2_project.data_utils import ensure_label_text_column, stratified_subset, unique_label_texts
from gliner2_project.ensemble import (
    MEAN_CONFIDENCE_ENSEMBLE,
    confidence_coverage,
    mean_confidence_ensemble,
)
from gliner2_project.env_utils import collect_environment_info, print_environment_info
from gliner2_project.gliner2_wrappers import load_gliner2_model, predict_one_classification
from gliner2_project.plotting import (
    plot_candidate_count_distribution,
    plot_candidate_recall_by_label,
    plot_metric_bar,
)
from gliner2_project.retrieval import (
    assert_no_test_ids_in_retriever,
    build_tfidf_retriever,
    prepare_examples,
    tfidf_knn_predict,
)
from gliner2_project.retrieval_analysis import (
    candidate_count_distribution,
    candidate_recall_analysis,
    gold_missing_candidate_examples,
    retrieval_confusion_pairs,
    retrieval_results_summary,
)
from gliner2_project.schema_variants import (
    BANKING_REQUEST_LABEL,
    CUSTOMER_INTENT_LABEL,
    ENSEMBLE_INPUT_METHODS,
    PLAIN_LABEL,
    QUERY_ABOUT_LABEL,
    build_schema_variant,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RUN_START_UTC = datetime.now(timezone.utc).isoformat()
environment_info = print_environment_info()
print(f"Run started at: {RUN_START_UTC}")

## Load GLiNER2

In [ ]:
if USE_GPU_IF_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

if DEVICE == "cpu":
    print("WARNING: GPU is unavailable or disabled. CPU inference can be slow.")
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

model = load_gliner2_model(MODEL_NAME, device=DEVICE)
print("Model loaded.")

## Load Banking77 train/test

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
print("Split names:", list(dataset.keys()))
if "train" not in dataset or "test" not in dataset:
    raise ValueError("Dataset must contain train and test splits.")

train_split = ensure_label_text_column(dataset["train"])
test_split = ensure_label_text_column(dataset["test"])
label_texts = unique_label_texts(test_split)
known_labels = set(label_texts)

print(f"Train examples: {len(train_split)}")
print(f"Test examples: {len(test_split)}")
print(f"Labels: {len(label_texts)}")

## Build evaluation subset

In [ ]:
def dataset_to_rows(split):
    rows = []
    for idx, row in enumerate(split):
        item = dict(row)
        item.setdefault("example_id", idx)
        rows.append(item)
    return rows


if MODE == "smoke":
    eval_examples = stratified_subset(
        test_split,
        per_label=1,
        seed=SEED,
        max_examples=SMOKE_N_EXAMPLES,
    )
elif MODE == "small":
    eval_examples = stratified_subset(
        test_split,
        per_label=SMALL_PER_LABEL,
        seed=SEED,
    )
elif MODE == "full":
    if not FULL_USE_ALL_TEST:
        raise ValueError("MODE='full' currently expects FULL_USE_ALL_TEST=True.")
    eval_examples = dataset_to_rows(test_split)
else:
    raise ValueError("MODE must be one of: smoke, small, full.")

example_ids = [example["example_id"] for example in eval_examples]
label_counts = pd.Series([example["label_text"] for example in eval_examples]).value_counts()
print(f"Mode: {MODE}")
print(f"Evaluation examples: {len(eval_examples)}")
print(f"Label coverage: {len(label_counts)} / {len(label_texts)}")
print(
    "Estimated GLiNER2 calls for plain_label + full mean ensemble + pruned GLiNER2 + pruned mean ensemble:",
    len(eval_examples) * (1 + len(ENSEMBLE_INPUT_METHODS) + 1 + len(ENSEMBLE_INPUT_METHODS)),
)

with (OUTPUT_PATH / "evaluated_example_ids.json").open("w", encoding="utf-8") as f:
    json.dump(example_ids, f, indent=2)

## Build TF-IDF retrieval index on train only

In [ ]:
train_examples = prepare_examples(train_split)
test_examples_for_check = [dict(row, example_id=idx) for idx, row in enumerate(test_split)]
retriever = build_tfidf_retriever(
    train_examples,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
)

# Banking77 train/test example IDs can overlap numerically because they are split-local.
# To avoid a false positive here, the retriever is built from train_split objects only,
# and test texts are never passed to build_tfidf_retriever.
print(f"TF-IDF train matrix shape: {retriever.train_matrix.shape}")

for example in eval_examples[:3]:
    print("=" * 80)
    print("query:", example["text"])
    for neighbor in retriever.retrieve(example["text"], k=3):
        print(neighbor["rank"], neighbor["label_text"], round(neighbor["similarity"], 4), neighbor["text"])

## Cache helpers

In [ ]:
def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def load_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=json_safe) + "\n")


BASE_MANIFEST = {
    "project_name": PROJECT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "mode": MODE,
    "example_ids": example_ids,
    "num_examples": len(eval_examples),
    "retrieval_params": {
        "tfidf_ngram_range": TFIDF_NGRAM_RANGE,
        "tfidf_min_df": TFIDF_MIN_DF,
        "tfidf_max_features": TFIDF_MAX_FEATURES,
        "k_neighbors": K_NEIGHBORS,
        "expanded_k_values": EXPANDED_K_VALUES,
        "min_candidates": MIN_CANDIDATES,
        "max_candidates": MAX_CANDIDATES,
    },
    "seed": SEED,
    "device": DEVICE,
}


def manifest_for(method_name):
    return {**BASE_MANIFEST, "method": method_name}


def cache_paths(method_name):
    return (
        PREDICTIONS_DIR / f"{method_name}.jsonl",
        PREDICTIONS_DIR / f"{method_name}.manifest.json",
    )


def load_cache(method_name):
    pred_path, manifest_path = cache_paths(method_name)
    if FORCE_RERUN or not pred_path.exists() or not manifest_path.exists():
        return None
    try:
        cached_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return None
    if cached_manifest == manifest_for(method_name):
        print(f"Loading cache for {method_name}: {pred_path}")
        return load_jsonl(pred_path)
    return None


def save_cache(method_name, rows):
    pred_path, manifest_path = cache_paths(method_name)
    save_jsonl(rows, pred_path)
    manifest_path.write_text(json.dumps(manifest_for(method_name), indent=2, default=json_safe), encoding="utf-8")

## Compute candidate sets and recall

In [ ]:
candidate_sets = {}
candidate_recall_rows = []
for example in tqdm(eval_examples, desc="candidate sets"):
    candidate_set = build_retrieved_candidate_set(
        query_text=example["text"],
        retriever=retriever,
        k_values=EXPANDED_K_VALUES,
        min_candidates=MIN_CANDIDATES,
        max_candidates=MAX_CANDIDATES,
        known_labels=known_labels,
    )
    candidate_sets[example["example_id"]] = candidate_set
    candidate_recall_rows.append(candidate_recall_row(example, candidate_set))

candidate_recall_df = pd.DataFrame(candidate_recall_rows)
candidate_recall_summary = candidate_recall_analysis(candidate_recall_rows)
candidate_counts = candidate_count_distribution(candidate_recall_rows)
gold_missing = gold_missing_candidate_examples(candidate_recall_rows)

candidate_recall_summary.to_csv(TABLES_DIR / "candidate_recall_analysis.csv", index=False)
candidate_counts.to_csv(TABLES_DIR / "candidate_count_distribution.csv", index=False)
gold_missing.to_csv(TABLES_DIR / "gold_missing_candidate_examples.csv", index=False)

display(candidate_recall_summary)
display(candidate_counts.head(20))
print("Gold missing examples:", len(gold_missing))

## Evaluation helpers

In [ ]:
def method_row_common(example, method_name, candidate_set=None):
    if candidate_set is None:
        return {
            "example_id": example["example_id"],
            "text": example["text"],
            "gold_label": example["label_text"],
            "method": method_name,
            "num_candidates": None,
            "gold_in_candidates": None,
            "retrieved_neighbor_ids": None,
            "retrieved_neighbor_labels": None,
            "retrieved_neighbor_similarities": None,
            "expanded_k_used": None,
        }
    candidates = candidate_set["candidate_labels"]
    return {
        "example_id": example["example_id"],
        "text": example["text"],
        "gold_label": example["label_text"],
        "method": method_name,
        "num_candidates": len(candidates),
        "gold_in_candidates": example["label_text"] in set(candidates),
        "retrieved_neighbor_ids": candidate_set["neighbor_ids"],
        "retrieved_neighbor_labels": candidate_set["neighbor_labels"],
        "retrieved_neighbor_similarities": candidate_set["neighbor_similarities"],
        "expanded_k_used": candidate_set["expanded_k_used"],
    }


def display_to_candidate_mapping(canonical_labels, variant_name=PLAIN_LABEL):
    candidates, mapping = build_schema_variant(list(canonical_labels), variant_name)
    return candidates, mapping


def run_gliner2_plain_method(method_name, canonical_label_provider):
    cached = load_cache(method_name)
    if cached is not None:
        return cached
    rows = []
    for example in tqdm(eval_examples, desc=method_name):
        canonical_labels, candidate_set = canonical_label_provider(example)
        candidate_labels, candidate_to_canonical = display_to_candidate_mapping(canonical_labels, PLAIN_LABEL)
        row = predict_one_classification(
            model=model,
            text=example["text"],
            candidate_labels=candidate_labels,
            candidate_to_canonical=candidate_to_canonical,
            method_name=method_name,
            example_id=example["example_id"],
            gold_label=example["label_text"],
        )
        row.update(method_row_common(example, method_name, candidate_set))
        row["predicted_display_label"] = row.pop("predicted_candidate")
        rows.append(row)
    save_cache(method_name, rows)
    return rows


def run_mean_confidence_method(method_name, canonical_label_provider):
    cached = load_cache(method_name)
    if cached is not None:
        return cached
    variant_predictions = {variant: [] for variant in ENSEMBLE_INPUT_METHODS}
    for variant in ENSEMBLE_INPUT_METHODS:
        for example in tqdm(eval_examples, desc=f"{method_name}:{variant}"):
            canonical_labels, candidate_set = canonical_label_provider(example)
            candidate_labels, candidate_to_canonical = display_to_candidate_mapping(canonical_labels, variant)
            row = predict_one_classification(
                model=model,
                text=example["text"],
                candidate_labels=candidate_labels,
                candidate_to_canonical=candidate_to_canonical,
                method_name=variant,
                example_id=example["example_id"],
                gold_label=example["label_text"],
            )
            row.update(method_row_common(example, variant, candidate_set))
            row["predicted_display_label"] = row.pop("predicted_candidate")
            variant_predictions[variant].append(row)
    if confidence_coverage(variant_predictions) == 0.0:
        print(f"Skipping {method_name}: confidence unavailable.")
        rows = []
    else:
        rows = mean_confidence_ensemble(variant_predictions)
        by_id_candidate = {
            example["example_id"]: candidate_sets.get(example["example_id"])
            for example in eval_examples
        }
        latency_by_example = {}
        for variant_rows in variant_predictions.values():
            for variant_row in variant_rows:
                latency_by_example.setdefault(variant_row["example_id"], 0.0)
                latency_by_example[variant_row["example_id"]] += float(
                    variant_row.get("latency_sec") or 0.0
                )
        for row in rows:
            example = next(item for item in eval_examples if item["example_id"] == row["example_id"])
            candidate_set = by_id_candidate.get(row["example_id"])
            if "candidate_pruning" not in method_name:
                candidate_set = None
            row.update(method_row_common(example, method_name, candidate_set))
            row["method"] = method_name
            row["latency_sec"] = latency_by_example.get(row["example_id"])
            row["raw_output"] = {
                "method_predictions": row.get("method_predictions"),
                "confidence_values": row.get("confidence_values"),
                "mean_confidence_by_label": row.get("mean_confidence_by_label"),
            }
            row["predicted_display_label"] = None
    save_cache(method_name, rows)
    return rows

## Evaluate methods

In [ ]:
ALL_CANONICAL_LABELS = list(label_texts)

predictions_by_method = {}

def all_labels_provider(example):
    return ALL_CANONICAL_LABELS, None


def pruned_labels_provider(example):
    candidate_set = candidate_sets[example["example_id"]]
    return candidate_set["candidate_labels"], candidate_set


method_name = "tfidf_knn_baseline"
cached = load_cache(method_name)
if cached is None:
    rows = []
    for example in tqdm(eval_examples, desc=method_name):
        start = time.perf_counter()
        result = tfidf_knn_predict(example["text"], retriever, K_NEIGHBORS)
        latency = time.perf_counter() - start
        base = method_row_common(example, method_name, candidate_sets[example["example_id"]])
        base.update(
            {
                "candidate_labels": list(result["label_scores"].keys()),
                "raw_output": {
                    "label_scores": result["label_scores"],
                    "neighbors": result["neighbors"],
                },
                "predicted_canonical": result["predicted_canonical"],
                "predicted_display_label": None,
                "confidence": None,
                "latency_sec": latency,
                "parse_error": None,
            }
        )
        rows.append(base)
    save_cache(method_name, rows)
else:
    rows = cached
predictions_by_method[method_name] = rows

predictions_by_method["plain_label"] = run_gliner2_plain_method(
    "plain_label",
    all_labels_provider,
)
predictions_by_method["mean_confidence_ensemble"] = run_mean_confidence_method(
    "mean_confidence_ensemble",
    all_labels_provider,
)
predictions_by_method["tfidf_candidate_pruning_gliner2"] = run_gliner2_plain_method(
    "tfidf_candidate_pruning_gliner2",
    pruned_labels_provider,
)
predictions_by_method["tfidf_candidate_pruning_mean_confidence_ensemble"] = run_mean_confidence_method(
    "tfidf_candidate_pruning_mean_confidence_ensemble",
    pruned_labels_provider,
)

print("Methods complete:", list(predictions_by_method))

## Compare all methods

In [ ]:
summary = retrieval_results_summary(predictions_by_method)
summary_path = TABLES_DIR / "retrieval_results_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary.sort_values("macro_f1", ascending=False).reset_index(drop=True))

per_label = per_label_metrics_frame(predictions_by_method)
per_label.to_csv(TABLES_DIR / "retrieval_per_label_metrics.csv", index=False)

confusions = retrieval_confusion_pairs(predictions_by_method)
confusions.to_csv(TABLES_DIR / "retrieval_confusion_pairs.csv", index=False)

improved, degraded = compare_method_examples(
    predictions_by_method,
    baseline_method="plain_label",
    comparison_method="tfidf_candidate_pruning_gliner2",
)
improved.to_csv(TABLES_DIR / "retrieval_improved_examples.csv", index=False)
degraded.to_csv(TABLES_DIR / "retrieval_degraded_examples.csv", index=False)

mean_improved, mean_degraded = compare_method_examples(
    predictions_by_method,
    baseline_method="mean_confidence_ensemble",
    comparison_method="tfidf_candidate_pruning_mean_confidence_ensemble",
)
mean_improved.to_csv(TABLES_DIR / "retrieval_mean_ensemble_improved_examples.csv", index=False)
mean_degraded.to_csv(TABLES_DIR / "retrieval_mean_ensemble_degraded_examples.csv", index=False)

print("plain wrong -> pruning GLiNER2 correct:", len(improved))
print("plain correct -> pruning GLiNER2 wrong:", len(degraded))
print("mean ensemble wrong -> pruning mean ensemble correct:", len(mean_improved))
print("mean ensemble correct -> pruning mean ensemble wrong:", len(mean_degraded))

## Candidate pruning analysis

In [ ]:
pruned_rows = predictions_by_method["tfidf_candidate_pruning_gliner2"]
pruned_frame = pd.DataFrame(pruned_rows)
gold_missing_predictions = pruned_frame[~pruned_frame["gold_in_candidates"].astype(bool)].copy()
gold_present_wrong = pruned_frame[
    pruned_frame["gold_in_candidates"].astype(bool)
    & (pruned_frame["gold_label"] != pruned_frame["predicted_canonical"])
].copy()
gold_missing_predictions.to_csv(TABLES_DIR / "gold_missing_candidate_examples.csv", index=False)
gold_present_wrong.to_csv(TABLES_DIR / "gold_in_candidates_but_wrong_examples.csv", index=False)

print("Gold missing from candidates:", len(gold_missing_predictions))
print("Gold in candidates but GLiNER2 wrong:", len(gold_present_wrong))
display(gold_missing_predictions.head(10))
display(gold_present_wrong.head(10))

## Visualizations

In [ ]:
figure_paths = []
figure_paths.append(
    plot_metric_bar(
        summary,
        "accuracy",
        FIGURES_DIR / "retrieval_method_comparison_accuracy.png",
        "Retrieval Method Comparison: Accuracy",
    )
)
figure_paths.append(
    plot_metric_bar(
        summary,
        "macro_f1",
        FIGURES_DIR / "retrieval_method_comparison_macro_f1.png",
        "Retrieval Method Comparison: Macro F1",
    )
)
figure_paths.append(
    plot_candidate_count_distribution(
        candidate_counts,
        FIGURES_DIR / "candidate_count_distribution.png",
    )
)
figure_paths.append(
    plot_candidate_recall_by_label(
        candidate_recall_df,
        FIGURES_DIR / "candidate_recall_by_label.png",
    )
)
for path in figure_paths:
    print(path)

## Save manifest

In [ ]:
RUN_END_UTC = datetime.now(timezone.utc).isoformat()
manifest = {
    **BASE_MANIFEST,
    "start_time_utc": RUN_START_UTC,
    "end_time_utc": RUN_END_UTC,
    "package_versions": environment_info,
    "methods": list(predictions_by_method.keys()),
    "artifacts": {
        "predictions_dir": str(PREDICTIONS_DIR),
        "tables_dir": str(TABLES_DIR),
        "figures_dir": str(FIGURES_DIR),
        "results_summary": str(TABLES_DIR / "retrieval_results_summary.csv"),
        "candidate_recall_analysis": str(TABLES_DIR / "candidate_recall_analysis.csv"),
    },
    "reporting_note": (
        "This retrieval extension uses Banking77 train as retrieval memory and is not pure zero-shot."
    ),
}
manifest_path = OUTPUT_PATH / "retrieval_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=json_safe), encoding="utf-8")
print(f"Saved {manifest_path}")

## Final short summary

In [ ]:
print("Retrieval candidate pruning experiment complete.")
print(f"Mode: {MODE}")
print(f"Examples: {len(eval_examples)}")
print("This method is retrieval-augmented, not pure zero-shot.")
display(summary.sort_values("macro_f1", ascending=False).reset_index(drop=True))
display(candidate_recall_summary)